In [2]:
import tensorflow as tf
from tensorflow.keras import backend

from tensorflow.keras import layers
import keras
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2


I0000 00:00:1774926491.896378  100118 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
def get_flops_tf_keras(model, input_shape=(1, 224, 224, 3)):
    @tf.function
    def model_fn(x):
        return model(x, training=False)

    concrete = model_fn.get_concrete_function(
            tf.TensorSpec(input_shape, tf.float32)
    )
    frozen = convert_variables_to_constants_v2(concrete)
    graph_def = frozen.graph.as_graph_def(add_shapes=True)

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")
        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        flops = tf.compat.v1.profiler.profile(
                graph=graph, run_meta=run_meta, cmd="op", options=opts
        )

    return flops.total_float_ops

# ShufflenNet V2

In [4]:
class ShuffleV2Block(tf.keras.layers.Layer):
    def __init__(self, inp, oup, mid_channels, *, ksize, stride):
        super().__init__()
        if stride not in (1, 2):
            raise ValueError("stride must be 1 or 2")

        self.stride = stride
        outputs = oup - inp

        self.branch_main = tf.keras.Sequential(
            [
                tf.keras.layers.Conv2D(mid_channels, 1, strides=1, padding="valid", use_bias=False),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.ReLU(),
                tf.keras.layers.DepthwiseConv2D(ksize, strides=stride, padding="same", use_bias=False),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.Conv2D(outputs, 1, strides=1, padding="valid", use_bias=False),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.ReLU(),
            ]
        )

        self.branch_proj = None
        if stride == 2:
            self.branch_proj = tf.keras.Sequential(
                [
                    tf.keras.layers.DepthwiseConv2D(ksize, strides=2, padding="same", use_bias=False),
                    tf.keras.layers.BatchNormalization(),
                    tf.keras.layers.Conv2D(inp, 1, strides=1, padding="valid", use_bias=False),
                    tf.keras.layers.BatchNormalization(),
                    tf.keras.layers.ReLU(),
                ]
            )

    def call(self, old_x, training=False):
        if self.stride == 1:
            x_proj, x = self.channel_shuffle(old_x)
            return tf.concat([x_proj, self.branch_main(x, training=training)], axis=-1)

        return tf.concat(
            [
                self.branch_proj(old_x, training=training),
                self.branch_main(old_x, training=training),
            ],
            axis=-1,
        )

    def channel_shuffle(self, x):
        shape = tf.shape(x)
        batch_size, height, width, num_channels = shape[0], shape[1], shape[2], shape[3]
        tf.debugging.assert_equal(num_channels % 4, 0, message="num_channels must be divisible by 4")

        x = tf.reshape(x, [batch_size, height, width, num_channels // 2, 2])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [batch_size, height, width, num_channels])
        return tf.split(x, num_or_size_splits=2, axis=-1)


class ShuffleNetV2(tf.keras.Model):
    def __init__(self, input_size=224, n_class=1000, model_size="1.5x"):
        super().__init__()
        print("model size is", model_size)

        self.stage_repeats = [4, 8, 4]
        self.model_size = model_size
        if model_size == "0.5x":
            self.stage_out_channels = [-1, 24, 48, 96, 192, 1024]
        elif model_size == "0.75x":
            self.stage_out_channels = [-1, 24, 108, 216, 432, 1024]
        elif model_size == "0.90x":
            self.stage_out_channels = [-1, 24, 108, 216, 448, 1024]
        elif model_size == "1.0x":
            self.stage_out_channels = [-1, 24, 116, 232, 464, 1024]
        elif model_size == "1.5x":
            self.stage_out_channels = [-1, 24, 176, 352, 704, 1024]
        elif model_size == "2.0x":
            self.stage_out_channels = [-1, 24, 244, 488, 976, 2048]
        else:
            raise NotImplementedError

        input_channel = self.stage_out_channels[1]
        self.first_conv = tf.keras.Sequential(
            [
                tf.keras.layers.Conv2D(input_channel, 3, strides=2, padding="same", use_bias=False),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.ReLU(),
            ]
        )
        self.maxpool = tf.keras.layers.MaxPool2D(pool_size=3, strides=2, padding="same")

        features = []
        for idxstage, numrepeat in enumerate(self.stage_repeats):
            output_channel = self.stage_out_channels[idxstage + 2]

            for i in range(numrepeat):
                if i == 0:
                    features.append(
                        ShuffleV2Block(
                            input_channel,
                            output_channel,
                            mid_channels=output_channel // 2,
                            ksize=3,
                            stride=2,
                        )
                    )
                else:
                    features.append(
                        ShuffleV2Block(
                            input_channel // 2,
                            output_channel,
                            mid_channels=output_channel // 2,
                            ksize=3,
                            stride=1,
                        )
                    )
                input_channel = output_channel

        self.features = tf.keras.Sequential(features)
        self.conv_last = tf.keras.Sequential(
            [
                tf.keras.layers.Conv2D(
                    self.stage_out_channels[-1],
                    1,
                    strides=1,
                    padding="valid",
                    use_bias=False,
                ),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.ReLU(),
            ]
        )

        self.globalpool = tf.keras.layers.GlobalAveragePooling2D()
        self.dropout = tf.keras.layers.Dropout(0.2) if self.model_size == "2.0x" else None
        self.classifier = tf.keras.layers.Dense(n_class, use_bias=False)

    def call(self, x, training=False):
        x = self.first_conv(x, training=training)
        x = self.maxpool(x)
        x = self.features(x, training=training)
        x = self.conv_last(x, training=training)

        x = self.globalpool(x)
        if self.dropout is not None:
            x = self.dropout(x, training=training)
        x = self.classifier(x)
        return x

In [5]:
shuffleNetV2 = ShuffleNetV2(model_size="0.90x", n_class=10)
shuffleNetV2_flops = get_flops_tf_keras(shuffleNetV2, input_shape=(1, 224, 224, 3))

print(f"ShuffleNetV2 FLOPS: {shuffleNetV2_flops / 10 ** 9:.03} G") # ShuffleNetV2 FLOPS: 0.262 G

model size is 0.90x


W0000 00:00:1774926517.610088  100118 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
I0000 00:00:1774926519.107956  100118 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1774926519.108051  100118 single_machine.cc:376] Starting new session
W0000 00:00:1774926519.110228  100118 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.

=========================Options=============================
ShuffleNetV2 FLOPS: 0.267 G
-max_depth                  10000
-min_bytes                  0
-min_peak_bytes             0
-min_residual_bytes         0
-min_output_bytes           0
-min_micros                 0
-min_accelerator_micros     0
-min_cpu_micros             0
-min_params                 0
-min_float_ops              1
-min_occurrence             0
-step                       -1
-order_by                   float_ops
-account_type_regexes       .*
-start_name_regexes         .*
-trim_name_regexes          
-show_name_regexes          .*
-hide_name_regexes          
-account_displayed_op_only  true
-select                     float_ops
-output                     stdout:

==================Model Analysis Report======================

Doc:
op:

In [ ]:
shuffleNetV2.summary()

Model: "shuffle_net_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (1, 112, 112, 24)      │           744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_20 (Sequential)      │ (1, 7, 7, 448)         │       717,820 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_21 (Sequential)      │ (1, 7, 7, 1024)        │       462,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 10)                │        10,240 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,191,652 (4.55 MB)

 Trainable params: 1,176,232 (4.49 MB)

 Non-trainable params: 15,420 (60.23 KB)

# Teacher - DenseNet201 and DenseNet-Mini (Baseline)

In [7]:
import tensorflow as tf
from tensorflow.keras import backend

from tensorflow.keras import layers

"""
## Create student and teacher models

Initialy, we create a teacher model and a smaller student model. Both models are
convolutional neural networks and created using `Sequential()`,
but could be any Keras model.
"""

input_size=(224, 224, 3)

####### DENSENET
dense_block_growth_rate = 32
dense_block_layers = 4
dense_kernel_size = 3
dense_reduction = 0.5

###### MOBILENET
mobilenet_alpha = 1


def DensePreAmble(x, postfix):
    bn_axis = 3 if backend.image_data_format() == "channels_last" else 1 # bn_axis is the batch_norm axis
    prefix = "DensePreAmble_" + postfix + "_"


    x = layers.ZeroPadding2D(padding=((3, 3), (3, 3)), name=prefix+"ZeroPadding2D_1")(x)
    x = layers.Conv2D(64, 7, strides=2, use_bias=False, name=prefix+"Conv2D_1")(x)
    x = layers.BatchNormalization(axis=bn_axis, epsilon=1.001e-5, name=prefix+"BatchNormalization")(x) # x = layers.LayerNormalization()(x) from ConvNeXt?
    x = layers.ReLU(name=prefix+"ReLU_1")(x) # x = layers.GeLU()(x) from ConvNeXt? or Swish from efficientnet?
    x = layers.ZeroPadding2D(padding=((1, 1), (1, 1)), name=prefix+"ZeroPadding2D_2")(x)
    x = layers.MaxPooling2D(3, strides=2, name=prefix+"MaxPooling2D")(x)
    return x

def DenseBlock(x, growth_rate, kernel_size, num_layers, postfix):
    prefix = "DenseBlock_" + postfix + "_"
    for i in range(num_layers):
        prefix_layer = prefix + str(i) + "_"
        dense_connection = x # stores the input so it can be concactinated
        
        bn_axis = 3 if backend.image_data_format() == "channels_last" else 1 # bn_axis is the batch_norm axis
        
        x = layers.BatchNormalization(axis=bn_axis, epsilon=1.001e-5, name=prefix_layer+"BatchNormalization")(x) # x = layers.LayerNormalization()(x) from ConvNeXt?
        x = layers.ReLU(name=prefix_layer+"ReLU_1")(x) # x = layers.GeLU()(x) from ConvNeXt? or Swish from efficientnet?

        x = layers.Conv2D(4 * growth_rate, 1, use_bias=False, name=prefix_layer+"Conv2D_Bottleneck")(x) # BottleNeck 1x1 Conv Layer
        x = layers.BatchNormalization(axis=bn_axis, epsilon=1.001e-5, name=prefix_layer+"BatchNormalization_2")(x) # x = layers.LayerNormalization()(x) from ConvNeXt?

        x = layers.ReLU(name=prefix_layer+"ReLU_2")(x) # x = layers.GeLU()(x) from ConvNeXt? or Swish from efficientnet?

        x = layers.Conv2D(growth_rate, kernel_size, padding="same", use_bias=False, name=prefix_layer+"Conv2D_2")(x) # Bigger kenel from ConvNeXt?

        x = layers.Concatenate(axis=bn_axis, name=prefix_layer+"Concatenate")([x, dense_connection]) # Concatenating the input to the Conv Layer to the output of the layer
    return x

def TransitionBlock(x, reduction, postfix):
    prefix = "TransitionBlock_" + postfix
    bn_axis = 3 if backend.image_data_format() == "channels_last" else 1
    x = layers.BatchNormalization(axis=bn_axis, epsilon=1.001e-5, name=prefix+"BatchNormalization")(x) # x = layers.LayerNormalization()(x) from ConvNeXt?
    x = layers.ReLU(name=prefix+"ReLU")(x)  # x = layers.GeLU()(x) from ConvNeXt? or Swish from efficientnet?
    x = layers.Conv2D(int(x.shape[bn_axis] * reduction), 1, use_bias=False, name=prefix+"Conv2D")(x)
    x = layers.AveragePooling2D(2, strides=2, name=prefix+"AveragePooling2D")(x)
    return x

def DensePostamble(x):
    bn_axis = 3 if backend.image_data_format() == "channels_last" else 1
    x = layers.BatchNormalization(axis=bn_axis, epsilon=1.001e-5, name="bn")(x)
    x = layers.Activation("relu", name="relu")(x)
    return x


## Teacher - Dense201

In [8]:
input = layers.Input((224,224,3))

preamble = DensePreAmble(input, "1")

model = DenseBlock(preamble, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=6, postfix="3")
model = TransitionBlock(model, dense_reduction, "5")

model = DenseBlock(model, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=12, postfix="7")
model = TransitionBlock(model, dense_reduction, "9")

model = DenseBlock(model, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=48, postfix="11")
model = TransitionBlock(model, dense_reduction, "13")

model = DenseBlock(model, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=32, postfix="15")
model = DensePostamble(model)


model = layers.GlobalAveragePooling2D()(model)
model = layers.Dense(10)(model)

teacher = keras.Model(inputs=input, outputs=model)

flops = get_flops_tf_keras(teacher, input_shape=(1, 224, 224, 3))
print(f"Teacher FLOPS: {flops / 10 ** 9:.03} G") # Teacher FLOPS: 8.63 G

I0000 00:00:1774934050.840587  100118 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1774934050.840656  100118 single_machine.cc:376] Starting new session
W0000 00:00:1774934050.841858  100118 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...



=========================Options=============================
-max_depth                  10000
-min_bytes                  0
-min_peak_bytes             0
-min_residual_bytes         0
-min_output_bytes           0
-min_micros                 0
-min_accelerator_micros     0
-min_cpu_micros             0
-min_params                 0
-min_float_ops              1
-min_occurrence             0
-step                       -1
-order_by                   float_ops
-account_type_regexes       .*
-start_name_regexes         .*
-trim_name_regexes          
-show_name_regexes          .*
-hide_name_regexes          
-account_displayed_op_only  true
-select                     float_ops
-output                     stdout:

==================Model Analysis Report======================
Teacher FLOPS: 8.63 G

Doc:
op: The nodes are operation kernel type, such as MatMul, Conv2D. Graph nodes belonging to the same type are aggregated together.
flops: Number of float operations. Note: Please read the

In [ ]:
teacher.summary()

#  Total params: 18,341,194 (69.97 MB)
#  Trainable params: 18,112,138 (69.09 MB)
#  Non-trainable params: 229,056 (894.75 KB)


Model: "functional_22"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_22      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ze… │ (None, 230, 230,  │          0 │ input_layer_22[0… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Co… │ (None, 112, 112,  │      9,408 │ DensePreAmble_1_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ba… │ (None, 112, 112,  │        256 │ DensePreAmble_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Re… │ (None, 112, 112,  │          0 │ DensePreAmble_1_… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ze… │ (None, 114, 114,  │          0 │ DensePreAmble_1_… │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ma… │ (None, 56, 56,    │          0 │ DensePreAmble_1_… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Bat… │ (None, 56, 56,    │        256 │ DensePreAmble_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_ReL… │ (None, 56, 56,    │          0 │ DenseBlock_3_0_B… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Con… │ (None, 56, 56,    │      8,192 │ DenseBlock_3_0_R… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Bat… │ (None, 56, 56,    │        512 │ DenseBlock_3_0_C… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_ReL… │ (None, 56, 56,    │          0 │ DenseBlock_3_0_B… │
│ (ReLU)              │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Con… │ (None, 56, 56,    │     36,864 │ DenseBlock_3_0_R… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Con… │ (None, 56, 56,    │          0 │ DenseBlock_3_0_C… │
│ (Concatenate)       │ 96)               │            │ DensePreAmble_1_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_1_Bat… │ (None, 56, 56,    │        384 │ DenseBlock_3_0_C… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_1_ReL… │ (None, 56, 56,    │          0 │ DenseBlock_3_1_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_1_Con… │ (None, 56, 56,    │     12,288 │ DenseBlock_3_1_R

 Total params: 18,341,194 (69.97 MB)

 Trainable params: 18,112,138 (69.09 MB)

 Non-trainable params: 229,056 (894.75 KB)

# Student Baseline - DenseNet Mini

In [10]:
input = layers.Input((224,224,3))

preamble = DensePreAmble(input, "1")

model = DenseBlock(preamble, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=2, postfix="3")
model = TransitionBlock(model, dense_reduction, "5")

model = DenseBlock(model, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=4, postfix="7")
model = TransitionBlock(model, dense_reduction, "9")

model = DenseBlock(model, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=8, postfix="11")
model = TransitionBlock(model, dense_reduction, "13")

model = DenseBlock(model, growth_rate=dense_block_growth_rate, kernel_size=dense_kernel_size, num_layers=4, postfix="15")
model = DensePostamble(model)


model = layers.GlobalAveragePooling2D()(model)
student_model = layers.Dense(10)(model)

student_model = tf.keras.Model(inputs=input, outputs=student_model)

flops = get_flops_tf_keras(student_model, input_shape=(1, 224, 224, 3))
print(f"Student FLOPS: {flops / 10 ** 9:.03} G") #Student FLOPS: 1.49 G

I0000 00:00:1774934165.385996  100118 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1774934165.386088  100118 single_machine.cc:376] Starting new session
W0000 00:00:1774934165.387461  100118 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...



=========================Options=============================
-max_depth                  10000
-min_bytes                  0
-min_peak_bytes             0
-min_residual_bytes         0
-min_output_bytes           0
-min_micros                 0
-min_accelerator_micros     0
-min_cpu_micros             0
-min_params                 0
-min_float_ops              1
-min_occurrence             0
-step                       -1
-order_by                   float_ops
-account_type_regexes       .*
-start_name_regexes         .*
-trim_name_regexes          
-show_name_regexes          .*
-hide_name_regexes          
-account_displayed_op_only  true
-select                     float_ops
-output                     stdout:

==================Model Analysis Report======================
Student FLOPS: 1.49 G

Doc:
op: The nodes are operation kernel type, such as MatMul, Conv2D. Graph nodes belonging to the same type are aggregated together.
flops: Number of float operations. Note: Please read the

In [ ]:
student_model.summary()

#  Total params: 1,196,138 (4.56 MB)
#  Trainable params: 1,183,114 (4.51 MB)
#  Non-trainable params: 13,024 (50.88 KB)


Model: "functional_23"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_23      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ze… │ (None, 230, 230,  │          0 │ input_layer_23[0… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Co… │ (None, 112, 112,  │      9,408 │ DensePreAmble_1_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ba… │ (None, 112, 112,  │        256 │ DensePreAmble_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Re… │ (None, 112, 112,  │          0 │ DensePreAmble_1_… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ze… │ (None, 114, 114,  │          0 │ DensePreAmble_1_… │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DensePreAmble_1_Ma… │ (None, 56, 56,    │          0 │ DensePreAmble_1_… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Bat… │ (None, 56, 56,    │        256 │ DensePreAmble_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_ReL… │ (None, 56, 56,    │          0 │ DenseBlock_3_0_B… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Con… │ (None, 56, 56,    │      8,192 │ DenseBlock_3_0_R… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Bat… │ (None, 56, 56,    │        512 │ DenseBlock_3_0_C… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_ReL… │ (None, 56, 56,    │          0 │ DenseBlock_3_0_B… │
│ (ReLU)              │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Con… │ (None, 56, 56,    │     36,864 │ DenseBlock_3_0_R… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_0_Con… │ (None, 56, 56,    │          0 │ DenseBlock_3_0_C… │
│ (Concatenate)       │ 96)               │            │ DensePreAmble_1_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_1_Bat… │ (None, 56, 56,    │        384 │ DenseBlock_3_0_C… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_1_ReL… │ (None, 56, 56,    │          0 │ DenseBlock_3_1_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ DenseBlock_3_1_Con… │ (None, 56, 56,    │     12,288 │ DenseBlock_3_1_R

 Total params: 1,196,138 (4.56 MB)

 Trainable params: 1,183,114 (4.51 MB)

 Non-trainable params: 13,024 (50.88 KB)